# Asymmetric Multilingual Acquisition: Typology-Aware Adaptive Mixing for BabyLMs

- Get HF_TOKEN with `write-access`

In [ ]:
# Cell 1 — Parameters (edit these)
# --------------------------------

# --- HuggingFace ---------------------------------------------------------
HF_NAMESPACE = "amosluna"      # personal account or org name
REPO_PREFIX  = "babylm-2026"    # all repos will be named ${HF_NAMESPACE}/${REPO_PREFIX}-${COND}-seed${SEED}
MAKE_PRIVATE = False           # MUST be False before final BabyLM submission (CFP requires public)

# --- Drive / repo layout --------------------------------------------------
DRIVE_ROOT  = "/content/drive/MyDrive/Researchs/BabyLM"
REPO_URL    = "https://github.com/Amos-Luna/Asymmetric-Multilingual-Acquisition_TAAM.git"
REPO_BRANCH = "main"
REPO_DIR    = "/content/taam"
TOKENIZER_PATH = "tokenizer/spm_32k_en_nl_zh.model"   # relative to REPO_DIR

# --- Which runs to upload -------------------------------------------------
RUNS_TO_UPLOAD = [
    ("2026-05-13_TAAM_seed42",    "TAAM",    42),
    ("2026-05-14_TAAM_seed1337",  "TAAM",    1337),
    ("2026-05-14_TAAM_v2_seed42", "TAAM_v2", 42),
    ("2026-05-14_TAAM_v2_seed1337","TAAM_v2", 1337),
    ("2026-05-15_B0_seed42",      "B0",      42),
]

# --- Behaviour flags ------------------------------------------------------
DRY_RUN     = False  # True = solo simula; False = sube de verdad a HuggingFace
ONLY_MAIN   = False  # if True, push only the final checkpoint (no chck_NM revisions)
VERBOSE     = True

print("Plan:")
for run_name, cond, seed in RUNS_TO_UPLOAD:
    repo_id = f"{HF_NAMESPACE}/{REPO_PREFIX}-{cond.lower()}-seed{seed}"
    print(f"  {run_name:35s}  →  {repo_id}")
print(f"\nDRY_RUN={DRY_RUN}, ONLY_MAIN={ONLY_MAIN}, MAKE_PRIVATE={MAKE_PRIVATE}")

In [ ]:
# Cell 2 — Mount Drive + clone repo + install HF SDK
# ------------------------------------------------

import os, sys, subprocess, shlex, time
from pathlib import Path

def run(cmd, **kw):
    print(f"$ {cmd}")
    return subprocess.run(cmd if isinstance(cmd, list) else shlex.split(cmd), check=True, **kw)

# Mount Drive (skip if already mounted)
if not Path(DRIVE_ROOT).is_dir():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print(f"WARN: could not mount Drive: {exc}")

# Clone the repo (idempotent)
if not Path(REPO_DIR).is_dir():
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    run(f"git -C {REPO_DIR} fetch --all")
    run(f"git -C {REPO_DIR} checkout {REPO_BRANCH}")
    run(f"git -C {REPO_DIR} pull --ff-only")

# Symlink heavy dirs from Drive (same as colab.ipynb).
# IMPORTANT: tokenizer/ has a .gitkeep tracked in git, so after `git clone` the
# folder exists in /content/taam/tokenizer/ with only that placeholder file.
# If we just do `if exists: skip`, the symlink is NEVER created and the script
# can't find spm_32k_en_nl_zh.model. Solution: move any placeholder content
# into Drive (where the real .model already lives), delete the repo's empty
# dir, then create the symlink.
import shutil
drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)
for sub in ("data/hf_cache", "data/tokens", "tokenizer", "runs", "wandb"):
    (drive_root / sub).mkdir(parents=True, exist_ok=True)
    dst = Path(REPO_DIR) / sub
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        for item in dst.iterdir():
            target = drive_root / sub / item.name
            if not target.exists():
                shutil.move(str(item), str(target))
        shutil.rmtree(dst)
    dst.symlink_to(drive_root / sub, target_is_directory=True)
    print(f"symlink: {dst} → {drive_root / sub}")

# Install minimal HF deps (very fast on Colab; usually already there)
run("pip install -q --upgrade huggingface_hub safetensors transformers sentencepiece")

# Quick sanity checks — read these BEFORE running cell 4.
tok = Path(REPO_DIR) / TOKENIZER_PATH
print(f"\nTokenizer path: {tok}")
print(f"  exists: {tok.is_file()}")
if not tok.is_file():
    print("  FIX: Drive/MyDrive/Researchs/BabyLM/tokenizer/spm_32k_en_nl_zh.model")
    print("       must exist. If missing, run scripts/train_tokenizer.py first.")

runs_sample = Path(REPO_DIR) / "runs" / RUNS_TO_UPLOAD[0][0]
print(f"\nSample run: {runs_sample}")
print(f"  exists: {runs_sample.is_dir()}")
ckpt_root = runs_sample / "checkpoints"
if ckpt_root.is_dir():
    all_ckpts = sorted(ckpt_root.iterdir())
    print(f"  checkpoints/ ({len(all_ckpts)} dirs)")
    print(f"  first 3: {[p.name for p in all_ckpts[:3]]}")
    print(f"  last 3:  {[p.name for p in all_ckpts[-3:]]}")
else:
    print("  FIX: no checkpoints/ — training may not have finished")

script = Path(REPO_DIR) / "scripts/upload_to_hf.py"
print(f"\nUpload script exists: {script.is_file()}")
if not script.is_file():
    print("  FIX: git pull in cell 2 should bring it; otherwise push to GitHub first")

print("\n✓ Environment ready.")

In [ ]:
# Cell 3 — Set HF token (from Colab Secrets if available)
# ----------------------------------------------------

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from Colab Secrets.")
except Exception as exc:
    print(f"(Colab Secrets not available: {exc})")

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from env.")

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste HF_TOKEN (hidden): ").strip()

assert HF_TOKEN, "HF_TOKEN is required."
os.environ["HF_TOKEN"] = HF_TOKEN

# Verify token by querying /api/whoami
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
me = api.whoami()
print(f"Authenticated as: {me.get('name', me.get('email', '<unknown>'))}")
print(f"Plan: {me.get('plan', '<unknown>')}")

In [ ]:
# Cell 4 — Run the upload pipeline for every (run, cond, seed)
# -----------------------------------------------------------

import os, subprocess, shlex
from pathlib import Path

os.chdir(REPO_DIR)
print(f"cwd = {os.getcwd()}")

errors = []
for run_name, cond, seed in RUNS_TO_UPLOAD:
    run_dir = Path("runs") / run_name
    if not run_dir.is_dir():
        print(f"\u2717 missing run dir: {run_dir} — skipping")
        errors.append((run_name, "missing"))
        continue
    repo_id = f"{HF_NAMESPACE}/{REPO_PREFIX}-{cond.lower()}-seed{seed}"
    cmd = [
        "python", "scripts/upload_to_hf.py",
        "--run-dir", str(run_dir),
        "--repo-id", repo_id,
        "--tokenizer", TOKENIZER_PATH,
        "--condition", cond,
        "--seed", str(seed),
        "--hf-token", HF_TOKEN,
    ]
    if DRY_RUN:    cmd.append("--dry-run")
    if ONLY_MAIN:  cmd.append("--only-main")
    if VERBOSE:    cmd.append("-v")
    if not MAKE_PRIVATE: pass  # default is public
    else: cmd.append("--private")

    print("\n" + "=" * 72)
    print(f"→ {run_name}  ({cond} seed={seed})  ⇒  {repo_id}")
    print("=" * 72)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print(f"✗ upload_to_hf failed for {run_name}: exit code {result.returncode}")
        errors.append((run_name, f"exit {result.returncode}"))
        continue
    print(f"✓ {run_name} OK")

print("\n" + "=" * 72)
if errors:
    print(f"DONE with {len(errors)} error(s):")
    for r, e in errors:
        print(f"  - {r}: {e}")
elif DRY_RUN:
    print("DONE — dry-run only (nothing on HuggingFace). Set DRY_RUN=False and re-run cell 4.")
else:
    print("DONE — all uploads completed. Check https://huggingface.co/" + HF_NAMESPACE)
print("=" * 72)